In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip -q "/content/drive/MyDrive/2413-Sanket-Naik-Dissertation-2025-26/KaggleDataset/ISICDataset.zip" -d "/content/data/"
print("Extrated files")

Extrated files


In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import cv2

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from torchvision import models

import albumentations as A
from albumentations.pytorch import ToTensorV2

from tqdm import tqdm
from sklearn.metrics import (
    accuracy_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, matthews_corrcoef, cohen_kappa_score,
    roc_curve
)

# =========================================================
# CONFIG
# =========================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CONFIG = {
    "batch_size": 32,
    "epochs": 10,
    "lr": 1e-4,
    "img_size": 224
}

INDEX_PATH = "/content/drive/MyDrive/2413-Sanket-Naik-Dissertation-2025-26/Implimentations/final_dataset_index.csv"
BASE_DIR = "/content/data/train-image/image"

RESNET_PATH = "resnet18_best.pth"
EFFICIENTNET_PATH = "efficientnet_best.pth"

# =========================================================
# HAIR REMOVAL
# =========================================================
def remove_hair(image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    kernel = cv2.getStructuringElement(1,(17,17))
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)
    _, thresh = cv2.threshold(blackhat,10,255,cv2.THRESH_BINARY)
    return cv2.inpaint(image, thresh, 1, cv2.INPAINT_TELEA)

# =========================================================
# DATASET
# =========================================================
class ISICDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(BASE_DIR, os.path.basename(row["path"]))

        image = np.array(Image.open(img_path).convert("RGB"))
        image = remove_hair(image)

        if self.transform:
            image = self.transform(image=image)["image"]

        label = torch.tensor(row["target"], dtype=torch.float32)
        return image, label

# =========================================================
# AUGMENTATION
# =========================================================
train_tf = A.Compose([
    A.Resize(224,224),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=25,p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.Normalize(),
    ToTensorV2()
])

val_tf = A.Compose([
    A.Resize(224,224),
    A.Normalize(),
    ToTensorV2()
])

# =========================================================
# LOAD DATA
# =========================================================
df = pd.read_csv(INDEX_PATH)

train_df = df[df["split"]=="train"]
val_df   = df[df["split"]=="val"]
test_df  = df[df["split"]=="test"]

# BALANCED SAMPLER
pos = train_df["target"].sum()
neg = len(train_df) - pos

weights = train_df["target"].apply(
    lambda x: 1/pos if x==1 else 1/neg
)

sampler = WeightedRandomSampler(weights.values, len(weights), replacement=True)

train_loader = DataLoader(ISICDataset(train_df, train_tf),
                          batch_size=CONFIG["batch_size"],
                          sampler=sampler)

val_loader = DataLoader(ISICDataset(val_df, val_tf),
                        batch_size=CONFIG["batch_size"],
                        shuffle=False)

test_loader = DataLoader(ISICDataset(test_df, val_tf),
                         batch_size=CONFIG["batch_size"],
                         shuffle=False)

# =========================================================
# FOCAL LOSS
# =========================================================
class FocalLoss(nn.Module):
    def __init__(self, gamma=2, alpha=0.75):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        probs = torch.sigmoid(logits)
        pt = torch.where(targets==1, probs, 1-probs)
        return (self.alpha * (1-pt)**self.gamma * bce).mean()

criterion = FocalLoss()

# =========================================================
# MODEL (ResNet18)
# =========================================================
model = models.resnet18(weights="IMAGENET1K_V1")
model.fc = nn.Linear(model.fc.in_features,1)
model = model.to(DEVICE)

optimizer = optim.Adam(model.parameters(), lr=CONFIG["lr"])

# =========================================================
# TRAIN
# =========================================================
best_auc = 0

for epoch in range(CONFIG["epochs"]):
    model.train()
    running_loss = 0

    for images, labels in tqdm(train_loader):
        images = images.to(DEVICE)
        labels = labels.unsqueeze(1).to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    # VALIDATION
    model.eval()
    y_true, y_prob = [], []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE)
            probs = torch.sigmoid(model(images)).cpu().numpy().flatten()
            y_prob.extend(probs)
            y_true.extend(labels.numpy())

    auc = roc_auc_score(y_true, y_prob)

    print(f"Epoch {epoch+1} | Loss {running_loss:.4f} | Val AUC {auc:.4f}")

    if auc > best_auc:
        best_auc = auc
        torch.save(model.state_dict(), RESNET_PATH)

# =========================================================
# LOAD BOTH MODELS (ENSEMBLE)
# =========================================================
model1 = models.efficientnet_b0(weights=None)
model1.classifier[1] = nn.Linear(model1.classifier[1].in_features,1)
model1.load_state_dict(torch.load(EFFICIENTNET_PATH))
model1 = model1.to(DEVICE)
model1.eval()

model2 = models.resnet18(weights=None)
model2.fc = nn.Linear(model2.fc.in_features,1)
model2.load_state_dict(torch.load(RESNET_PATH))
model2 = model2.to(DEVICE)
model2.eval()

# =========================================================
# FIND THRESHOLD (VAL)
# =========================================================
y_true, y_prob = [], []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(DEVICE)

        p1 = torch.sigmoid(model1(images))
        p2 = torch.sigmoid(model2(images))

        probs = ((p1+p2)/2).cpu().numpy().flatten()

        y_prob.extend(probs)
        y_true.extend(labels.numpy())

fpr, tpr, thresholds = roc_curve(y_true, y_prob)
best_threshold = thresholds[np.argmax(tpr - fpr)]

print("Best threshold:", best_threshold)

# =========================================================
# TEST (ENSEMBLE)
# =========================================================
y_true, y_pred, y_prob = [], [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)

        p1 = torch.sigmoid(model1(images))
        p2 = torch.sigmoid(model2(images))

        probs = ((p1+p2)/2).cpu().numpy().flatten()
        preds = (probs >= best_threshold).astype(int)

        y_prob.extend(probs)
        y_pred.extend(preds)
        y_true.extend(labels.numpy())

# =========================================================
# METRICS
# =========================================================
acc = accuracy_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob)
mcc = matthews_corrcoef(y_true, y_pred)
kappa = cohen_kappa_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)

print("\nENSEMBLE RESULTS")
print("Accuracy:", acc)
print("Recall:", rec)
print("F1:", f1)
print("AUROC:", auc)
print("MCC:", mcc)
print("Kappa:", kappa)
print("Confusion Matrix:\n", cm)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 237MB/s]
100%|██████████| 9330/9330 [24:12<00:00,  6.42it/s]


Epoch 1 | Loss 131.8594 | Val AUC 0.9265


100%|██████████| 9330/9330 [24:09<00:00,  6.44it/s]


Epoch 2 | Loss 39.7257 | Val AUC 0.9039


...

100%|██████████| 9330/9330 [24:00<00:00,  6.48it/s]


Epoch 9 | Loss 8.0665 | Val AUC 0.8696


100%|██████████| 9330/9330 [23:53<00:00,  6.51it/s]


Epoch 10 | Loss 8.2555 | Val AUC 0.8829


FileNotFoundError: [Errno 2] No such file or directory: 'efficientnet_best.pth'

In [ ]:
# =========================================================
# LOAD BOTH MODELS (ENSEMBLE)
# =========================================================
model1 = models.efficientnet_b0(weights=None)
model1.classifier[1] = nn.Linear(model1.classifier[1].in_features,1)
model1.load_state_dict(torch.load(EFFICIENTNET_PATH))
model1 = model1.to(DEVICE)
model1.eval()

model2 = models.resnet18(weights=None)
model2.fc = nn.Linear(model2.fc.in_features,1)
model2.load_state_dict(torch.load(RESNET_PATH))
model2 = model2.to(DEVICE)
model2.eval()

# =========================================================
# FIND THRESHOLD (VAL)
# =========================================================
y_true, y_prob = [], []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(DEVICE)

        p1 = torch.sigmoid(model1(images))
        p2 = torch.sigmoid(model2(images))

        probs = ((p1+p2)/2).cpu().numpy().flatten()

        y_prob.extend(probs)
        y_true.extend(labels.numpy())

fpr, tpr, thresholds = roc_curve(y_true, y_prob)
best_threshold = thresholds[np.argmax(tpr - fpr)]

print("Best threshold:", best_threshold)

# =========================================================
# TEST (ENSEMBLE)
# =========================================================
y_true, y_pred, y_prob = [], [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)

        p1 = torch.sigmoid(model1(images))
        p2 = torch.sigmoid(model2(images))

        probs = ((p1+p2)/2).cpu().numpy().flatten()
        preds = (probs >= best_threshold).astype(int)

        y_prob.extend(probs)
        y_pred.extend(preds)
        y_true.extend(labels.numpy())

# =========================================================
# METRICS
# =========================================================
acc = accuracy_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob)
mcc = matthews_corrcoef(y_true, y_pred)
kappa = cohen_kappa_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)

print("\nENSEMBLE RESULTS")
print("Accuracy:", acc)
print("Recall:", rec)
print("F1:", f1)
print("AUROC:", auc)
print("MCC:", mcc)
print("Kappa:", kappa)
print("Confusion Matrix:\n", cm)

Best threshold: 0.042691637

ENSEMBLE RESULTS
Accuracy: 0.8886433394720687
Recall: 0.7692307692307693
F1: 0.01804402742692169
AUROC: 0.9100372754998306
MCC: 0.07602481962258273
Kappa: 0.015455740151625097
Confusion Matrix:
 [[43378  5427]
 [   15    50]]


In [ ]:
# =========================================================
# LOAD BOTH MODELS (ENSEMBLE)
# =========================================================
model1 = models.efficientnet_b0(weights=None)
model1.classifier[1] = nn.Linear(model1.classifier[1].in_features,1)
model1.load_state_dict(torch.load(EFFICIENTNET_PATH))
model1 = model1.to(DEVICE)
model1.eval()

model2 = models.resnet18(weights=None)
model2.fc = nn.Linear(model2.fc.in_features,1)
model2.load_state_dict(torch.load(RESNET_PATH))
model2 = model2.to(DEVICE)
model2.eval()

# =========================================================
# FIND THRESHOLD (VAL)
# =========================================================
from sklearn.metrics import roc_curve

fpr, tpr, thresholds = roc_curve(y_true, y_prob)

specificity = 1 - fpr

# enforce 90% specificity
target_spec = 0.90

idx = np.where(specificity >= target_spec)[0][-1]
best_threshold = thresholds[idx]

print("New threshold:", best_threshold)

# =========================================================
# TEST (ENSEMBLE)
# =========================================================
y_true, y_pred, y_prob = [], [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)

        p1 = torch.sigmoid(model1(images))
        p2 = torch.sigmoid(model2(images))

        probs = ((p1+p2)/2).cpu().numpy().flatten()
        preds = (probs >= best_threshold).astype(int)

        y_prob.extend(probs)
        y_pred.extend(preds)
        y_true.extend(labels.numpy())

# =========================================================
# METRICS
# =========================================================
acc = accuracy_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob)
mcc = matthews_corrcoef(y_true, y_pred)
kappa = cohen_kappa_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)

print("\nENSEMBLE RESULTS")
print("Accuracy:", acc)
print("Recall:", rec)
print("F1:", f1)
print("AUROC:", auc)
print("MCC:", mcc)
print("Kappa:", kappa)
print("Confusion Matrix:\n", cm)

New threshold: 0.050131753

ENSEMBLE RESULTS
Accuracy: 0.9011868221812973
Recall: 0.7384615384615385
F1: 0.01949238578680203
AUROC: 0.9100372754998306
MCC: 0.07792618761324911
Kappa: 0.01691176884099954
Confusion Matrix:
 [[43993  4812]
 [   17    48]]


In [ ]:
# =========================================================
# LOAD BOTH MODELS (ENSEMBLE)
# =========================================================
model1 = models.efficientnet_b0(weights=None)
model1.classifier[1] = nn.Linear(model1.classifier[1].in_features,1)
model1.load_state_dict(torch.load(EFFICIENTNET_PATH))
model1 = model1.to(DEVICE)
model1.eval()

model2 = models.resnet18(weights=None)
model2.fc = nn.Linear(model2.fc.in_features,1)
model2.load_state_dict(torch.load(RESNET_PATH))
model2 = model2.to(DEVICE)
model2.eval()

# =========================================================
# FIND THRESHOLD (VAL)
# =========================================================
from sklearn.metrics import roc_curve

fpr, tpr, thresholds = roc_curve(y_true, y_prob)

specificity = 1 - fpr

# enforce 90% specificity
target_spec = 0.95

idx = np.where(specificity >= target_spec)[0][-1]
best_threshold = thresholds[idx]

print("New threshold:", best_threshold)

# =========================================================
# TEST (ENSEMBLE)
# =========================================================
y_true, y_pred, y_prob = [], [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)

        p1 = torch.sigmoid(model1(images))
        p2 = torch.sigmoid(model2(images))

        probs = ((p1+p2)/2).cpu().numpy().flatten()
        preds = (probs >= best_threshold).astype(int)

        y_prob.extend(probs)
        y_pred.extend(preds)
        y_true.extend(labels.numpy())

# =========================================================
# METRICS
# =========================================================
acc = accuracy_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob)
mcc = matthews_corrcoef(y_true, y_pred)
kappa = cohen_kappa_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)

print("\nENSEMBLE RESULTS")
print("Accuracy:", acc)
print("Recall:", rec)
print("F1:", f1)
print("AUROC:", auc)
print("MCC:", mcc)
print("Kappa:", kappa)
print("Confusion Matrix:\n", cm)

New threshold: 0.13268462

ENSEMBLE RESULTS
Accuracy: 0.9530795989359525
Recall: 0.6153846153846154
F1: 0.0337126000842815
AUROC: 0.9100372754998306
MCC: 0.09774661290485097
Kappa: 0.03120608407805503
Confusion Matrix:
 [[46537  2268]
 [   25    40]]
